# Arabic Sign Language Recognition (KARSL-502) - PyTorch Version

## Project Overview
This notebook implements an **Arabic Sign Language Recognition pipeline** using:

- **Normalized MediaPipe keypoints**
- **Conv1D temporal feature extraction**
- **BiLSTM sequence modeling**
- **Self-Attention mechanism**
- **PyTorch**

The workflow covers:

1. Environment setup
2. Dataset configuration
3. Data preprocessing
4. Keypoint normalization
5. Dataset preparation
6. Model architecture
7. Training pipeline
8. Evaluation and visualization
9. Inference utilities

---

## Deep Learning Architecture

The model combines:

- **Conv1D** for local temporal feature extraction
- **Bidirectional LSTM** for sequence understanding
- **Multi-Head Self-Attention** for contextual representation learning
- **Dense classification head** for final prediction

---

## Best Practices Applied

- Reproducible random seeds
- Modular helper functions
- Clear section separation
- Structured configuration cells
- Documented preprocessing pipeline
- Readable training workflow

# 1. Environment Setup

This section installs and imports all required dependencies for:

- Computer Vision
- Deep Learning
- Data Processing
- Visualization
- Sequence Modeling

> Best Practice: Keep all imports centralized to simplify debugging and environment management.

In [ ]:
# ===== 1) Install dependencies =====
!pip -q install mediapipe torch

In [ ]:
import os
import json
import random
import urllib.request
from collections import Counter, deque

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 2. Configuration

This section contains all configurable parameters used throughout the notebook.

### Includes
- Dataset paths
- Sign ID ranges
- Training/validation split
- Random seed
- Sequence length
- Batch size
- Model hyperparameters

> Best Practice: Centralizing configuration improves reproducibility and experiment tracking.

In [ ]:
DATA_PATH = "/kaggle/input/datasets/yousefdotpy/blablabla/karsl-502"
LABELS_XLSX = "/kaggle/input/datasets/yousefdotpy/kars502labels/KARSL-502_Labels.xlsx"

SIGN_ID_START = 289
SIGN_ID_END = 400

TRAIN_SIGNERS = ["01", "02"]
TEST_SIGNERS = ["01", "02"]

FRAMES_PER_SEQUENCE = 48
BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SIZE = 0.2

# 3. Load Label Metadata

The label metadata is loaded from the official KARSL-502 annotations file.

### Goals
- Read label mappings
- Filter selected sign IDs
- Prepare class dictionaries

This step ensures consistency between:
- dataset samples
- labels
- training targets

In [ ]:
karsl_df = pd.read_excel(LABELS_XLSX)
selected_signids = [str(i).zfill(4) for i in range(SIGN_ID_START, SIGN_ID_END + 1)]

labels_df = (
    karsl_df[karsl_df["SignID"].astype(str).str.zfill(4).isin(selected_signids)]
    .copy()
    .reset_index(drop=True)
)
labels_df["SignID_str"] = labels_df["SignID"].astype(str).str.zfill(4)

WORDS = labels_df["Sign-Arabic"].tolist()
LABEL_MAP = {word: idx for idx, word in enumerate(WORDS)}
SIGNID_TO_ARABIC = dict(zip(labels_df["SignID_str"], labels_df["Sign-Arabic"]))
ARABIC_TO_ENGLISH = dict(zip(labels_df["Sign-Arabic"], labels_df["Sign-English"]))

NUM_CLASSES = len(WORDS)
print("Number of classes:", NUM_CLASSES)
labels_df.head()

# 4. Preprocessing & Helper Functions

This section defines reusable utility functions for:

- Landmark normalization
- Coordinate adjustments
- Sequence padding/truncation
- Feature engineering
- Dataset preparation

---

## Keypoint Structure

Each frame contains:

| Component | Shape |
|---|---|
| Pose landmarks | `33 × 3` |
| Left hand landmarks | `21 × 3` |
| Right hand landmarks | `21 × 3` |

The preprocessing pipeline converts all landmarks into normalized feature vectors suitable for temporal deep learning models.

In [ ]:
def adjust_landmarks(arr, center):
    arr_reshaped = arr.reshape(-1, 3)
    center_repeated = np.tile(center, (len(arr_reshaped), 1))
    arr_adjusted = arr_reshaped - center_repeated
    return arr_adjusted.reshape(-1)

def normalize_single_frame(pose_flat, lh_flat, rh_flat):
    pose = pose_flat.reshape(-1, 3).copy()
    lh = lh_flat.reshape(-1, 3).copy()
    rh = rh_flat.reshape(-1, 3).copy()

    # Pose center: midpoint of shoulders if available, else nose.
    left_shoulder = pose[11]
    right_shoulder = pose[12]
    shoulders_valid = np.any(left_shoulder != 0.0) and np.any(right_shoulder != 0.0)

    if shoulders_valid:
        pose_center = (left_shoulder + right_shoulder) / 2.0
        pose_scale = np.linalg.norm(left_shoulder - right_shoulder)
    else:
        pose_center = pose[0]
        pose_scale = 1.0

    pose = pose - pose_center
    pose = pose / max(float(pose_scale), 1e-6)

    def norm_hand(hand):
        wrist = hand[0].copy()
        if np.any(wrist != 0.0):
            hand = hand - wrist
            palm_ref = hand[9]
            scale = np.linalg.norm(palm_ref) if np.any(palm_ref != 0.0) else 1.0
            hand = hand / max(float(scale), 1e-6)
        return hand

    lh = norm_hand(lh)
    rh = norm_hand(rh)

    return np.concatenate([pose.reshape(-1), lh.reshape(-1), rh.reshape(-1)], axis=0).astype(np.float32)

def normalize_sequence(pose_seq, lh_seq, rh_seq):
    frames = []
    for i in range(len(pose_seq)):
        frames.append(normalize_single_frame(pose_seq[i], lh_seq[i], rh_seq[i]))
    return np.asarray(frames, dtype=np.float32)

def resample_sequence(seq, target_len=48):
    if len(seq) == target_len:
        return seq.astype(np.float32)
    if len(seq) == 1:
        return np.repeat(seq, target_len, axis=0).astype(np.float32)

    old_idx = np.linspace(0, len(seq) - 1, num=len(seq))
    new_idx = np.linspace(0, len(seq) - 1, num=target_len)

    out = np.zeros((target_len, seq.shape[1]), dtype=np.float32)
    for d in range(seq.shape[1]):
        out[:, d] = np.interp(new_idx, old_idx, seq[:, d])
    return out

# 5. Dataset Construction

This stage builds the training and testing datasets directly from the official saved `.npy` keypoint arrays.

### Pipeline Steps
1. Load sequence arrays
2. Normalize landmarks
3. Resize sequences to fixed length
4. Generate labels
5. Convert to NumPy arrays

> Best Practice: Sequence normalization improves model stability and generalization.

In [ ]:
def build_dataset_from_official_keypoints(data_path, signers, split_name, frames_per_sequence=48):
    X, y, meta = [], [], []

    for signer in signers:
        signer_root = os.path.join(data_path, signer, split_name)
        if not os.path.isdir(signer_root):
            print(f"Missing path: {signer_root}")
            continue

        for signid in sorted(os.listdir(signer_root)):
            if signid not in SIGNID_TO_ARABIC:
                continue

            sign_root = os.path.join(signer_root, signid)
            lh_dir = os.path.join(sign_root, "lh_keypoints")
            rh_dir = os.path.join(sign_root, "rh_keypoints")
            pose_dir = os.path.join(sign_root, "pose_keypoints")

            if not (os.path.isdir(lh_dir) and os.path.isdir(rh_dir) and os.path.isdir(pose_dir)):
                continue

            lh_files = sorted([f for f in os.listdir(lh_dir) if f.endswith(".npy")])

            arabic_label = SIGNID_TO_ARABIC[signid]
            class_idx = LABEL_MAP[arabic_label]

            for fname in lh_files:
                lh_path = os.path.join(lh_dir, fname)
                rh_path = os.path.join(rh_dir, fname)
                pose_path = os.path.join(pose_dir, fname)

                if not (os.path.exists(lh_path) and os.path.exists(rh_path) and os.path.exists(pose_path)):
                    continue

                lh_seq = np.load(lh_path).astype(np.float32)
                rh_seq = np.load(rh_path).astype(np.float32)
                pose_seq = np.load(pose_path).astype(np.float32)

                n = min(len(lh_seq), len(rh_seq), len(pose_seq))
                if n == 0:
                    continue

                lh_seq = lh_seq[:n]
                rh_seq = rh_seq[:n]
                pose_seq = pose_seq[:n]

                seq = normalize_sequence(pose_seq, lh_seq, rh_seq)
                seq = resample_sequence(seq, target_len=frames_per_sequence)

                X.append(seq)
                y.append(class_idx)
                meta.append({
                    "signer": signer,
                    "split": split_name,
                    "signid": signid,
                    "sequence": fname,
                    "arabic": arabic_label,
                })

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64), meta

# 6. Train / Validation Split

The dataset is split into:
- Training set
- Validation set
- Testing set

### Why Validation Matters
The validation set helps:
- monitor overfitting
- tune hyperparameters
- evaluate generalization during training

In [ ]:
X_train_full, y_train_full, train_meta = build_dataset_from_official_keypoints(
    DATA_PATH, TRAIN_SIGNERS, "train", frames_per_sequence=FRAMES_PER_SEQUENCE
)
X_test, y_test, test_meta = build_dataset_from_official_keypoints(
    DATA_PATH, TEST_SIGNERS, "test", frames_per_sequence=FRAMES_PER_SEQUENCE
)

print("Train full:", X_train_full.shape, y_train_full.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train_full
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)

# 7. Model Architecture

The neural network combines several modern sequence learning techniques.

## Architecture Components

### Conv1D Layer
Extracts short-term temporal motion patterns.

### Bidirectional LSTM
Learns sequential dependencies from both directions.

### Multi-Head Self-Attention
Allows the model to focus on important temporal regions.

### Dense Classification Head
Produces final sign class probabilities.

---

## Design Goals
- Better temporal understanding
- Stronger contextual learning
- Improved sequence representation

In [ ]:
class ArabicSignModel(nn.Module):
    def __init__(self, num_classes, timesteps, feature_dim):
        super(ArabicSignModel, self).__init__()
        
        # Conv1D layers
        self.conv1 = nn.Conv1d(feature_dim, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.2)
        
        self.conv2 = nn.Conv1d(128, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.2)
        
        # Bidirectional LSTM layers
        self.lstm1 = nn.LSTM(128, 128, batch_first=True, bidirectional=True, dropout=0.25)
        self.lstm2 = nn.LSTM(256, 128, batch_first=True, bidirectional=True, dropout=0.25)
        
        # Multi-Head Attention
        self.attention = nn.MultiheadAttention(
            embed_dim=256,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )
        self.attn_norm = nn.LayerNorm(256)
        
        # Dense layers
        self.fc1 = nn.Linear(256 * 2, 256)  # 256*2 because we have global avg and max pooling
        self.bn3 = nn.BatchNorm1d(256)
        self.dropout3 = nn.Dropout(0.35)
        
        self.fc2 = nn.Linear(256, 128)
        self.dropout4 = nn.Dropout(0.25)
        
        self.fc_out = nn.Linear(128, num_classes)
        
        self.timesteps = timesteps
    
    def forward(self, x):
        # x shape: (batch, timesteps, features)
        # Conv1d expects (batch, features, timesteps)
        x = x.transpose(1, 2)
        
        # Conv1D blocks
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.nn.functional.silu(x)  # swish activation
        x = self.dropout1(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout2(x)
        
        # Back to (batch, timesteps, features) for LSTM
        x = x.transpose(1, 2)
        
        # Bidirectional LSTM
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        
        # Multi-Head Attention
        attn_out, _ = self.attention(x, x, x)
        x = x + attn_out
        x = self.attn_norm(x)
        
        # Global pooling
        avg_pool = torch.mean(x, dim=1)
        max_pool, _ = torch.max(x, dim=1)
        x = torch.cat([avg_pool, max_pool], dim=1)
        
        # Dense head
        x = self.fc1(x)
        x = self.bn3(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout3(x)
        
        x = self.fc2(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout4(x)
        
        x = self.fc_out(x)
        return x

# Build model
model = ArabicSignModel(NUM_CLASSES, X_train.shape[1], X_train.shape[2])
model.to(DEVICE)

# Print model architecture
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create data loaders
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.LongTensor(y_val)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Compute class weights
classes = np.unique(y_train)
class_weights_arr = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights_tensor = torch.FloatTensor(class_weights_arr).to(DEVICE)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(f"Class weights: {class_weights_tensor}")

# 8. Training Loop

In [ ]:
# Training utilities
best_val_acc = 0
patience = 12
patience_counter = 0
history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

# Training loop
best_val_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
          f"Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_arabic_sign_model.pt")
        print(f"  → Best model saved! Val Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("\nTraining complete!")

In [ ]:
# Plot training history
plt.figure(figsize=(8, 4))
plt.plot(history["loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.legend()
plt.title("Loss")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["accuracy"], label="train_accuracy")
plt.plot(history["val_accuracy"], label="val_accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()

# 9. Evaluation & Metrics

This section evaluates the trained model using:

- Accuracy
- Confusion matrix
- Classification metrics
- Training curves

The goal is to measure:
- generalization quality
- class-level performance
- model robustness

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_arabic_sign_model.pt"))

# Evaluate on test set
model.eval()
test_loss = 0
correct = 0
total = 0
all_preds = []
all_probs = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        test_loss += loss.item() * X_batch.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

avg_test_loss = test_loss / total
test_acc = correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print()
print(classification_report(y_test, all_preds, target_names=WORDS, digits=4, zero_division=0))

In [ ]:
# Save final model
torch.save(model.state_dict(), "final_arabic_sign_model.pt")

with open("label_map_arabic.json", "w", encoding="utf-8") as f:
    json.dump({str(i): word for i, word in enumerate(WORDS)}, f, ensure_ascii=False, indent=2)

print("Saved model and labels.")

# Deployment / Real-time Inference

## Download MediaPipe Tasks models for deployment / real-time inference

Training uses the official saved dataset keypoints.

The downloads below are only for **real-time inference** on new webcam/video frames using the newer MediaPipe Tasks API.

In [ ]:
os.makedirs("/kaggle/working/mediapipe_tasks", exist_ok=True)

HAND_TASK_PATH = "/kaggle/working/mediapipe_tasks/hand_landmarker.task"
POSE_TASK_PATH = "/kaggle/working/mediapipe_tasks/pose_landmarker_lite.task"

downloads = {
    HAND_TASK_PATH: "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    POSE_TASK_PATH: "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
}

for out_path, url in downloads.items():
    if not os.path.exists(out_path):
        print("Downloading:", url)
        urllib.request.urlretrieve(url, out_path)

print("Done.")
print(HAND_TASK_PATH, os.path.exists(HAND_TASK_PATH))
print(POSE_TASK_PATH, os.path.exists(POSE_TASK_PATH))

## Real-time inference with modern MediaPipe Tasks API

This part is for deployment/testing on new videos or locally with a webcam.

It maps the extracted pose/hand landmarks into the same feature format used during training.

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

In [ ]:
class MediaPipeTasksExtractor:
    def __init__(self, hand_task_path, pose_task_path):
        hand_base_options = mp_python.BaseOptions(model_asset_path=hand_task_path)
        pose_base_options = mp_python.BaseOptions(model_asset_path=pose_task_path)

        self.hand_landmarker = mp_vision.HandLandmarker.create_from_options(
            mp_vision.HandLandmarkerOptions(
                base_options=hand_base_options,
                num_hands=2,
                min_hand_detection_confidence=0.4,
                min_hand_presence_confidence=0.4,
                min_tracking_confidence=0.4,
            )
        )

        self.pose_landmarker = mp_vision.PoseLandmarker.create_from_options(
            mp_vision.PoseLandmarkerOptions(
                base_options=pose_base_options,
                output_segmentation_masks=False,
                num_poses=1,
                min_pose_detection_confidence=0.4,
                min_pose_presence_confidence=0.4,
                min_tracking_confidence=0.4,
            )
        )

    def close(self):
        self.hand_landmarker.close()
        self.pose_landmarker.close()

    @staticmethod
    def _to_mp_image(frame_bgr):
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    def extract_frame_features(self, frame_bgr):
        image = self._to_mp_image(frame_bgr)
        hand_result = self.hand_landmarker.detect(image)
        pose_result = self.pose_landmarker.detect(image)

        pose_xyz = np.zeros((33, 3), dtype=np.float32)
        left_hand_xyz = np.zeros((21, 3), dtype=np.float32)
        right_hand_xyz = np.zeros((21, 3), dtype=np.float32)

        if pose_result.pose_landmarks and len(pose_result.pose_landmarks) > 0:
            pose_landmarks = pose_result.pose_landmarks[0]
            for i, lm in enumerate(pose_landmarks[:33]):
                pose_xyz[i] = [lm.x, lm.y, lm.z]

        if hand_result.hand_landmarks:
            for idx, hand_lms in enumerate(hand_result.hand_landmarks):
                handedness_name = None
                if hand_result.handedness and idx < len(hand_result.handedness):
                    if len(hand_result.handedness[idx]) > 0:
                        handedness_name = hand_result.handedness[idx][0].category_name.lower()

                coords = np.zeros((21, 3), dtype=np.float32)
                for i, lm in enumerate(hand_lms[:21]):
                    coords[i] = [lm.x, lm.y, lm.z]

                if handedness_name == "left":
                    left_hand_xyz = coords
                elif handedness_name == "right":
                    right_hand_xyz = coords

        return normalize_single_frame(
            pose_xyz.reshape(-1),
            left_hand_xyz.reshape(-1),
            right_hand_xyz.reshape(-1)
        )

In [ ]:
class RealTimeSignPredictor:
    def __init__(
        self,
        pytorch_model_path="final_arabic_sign_model.pt",
        label_map_json_path="label_map_arabic.json",
        hand_task_path=HAND_TASK_PATH,
        pose_task_path=POSE_TASK_PATH,
        frames_per_sequence=48,
        smoothing_window=8,
        min_confidence=0.65,
    ):
        # Load PyTorch model
        self.model = ArabicSignModel(NUM_CLASSES, frames_per_sequence, 195)
        self.model.load_state_dict(torch.load(pytorch_model_path, map_location=DEVICE))
        self.model.to(DEVICE)
        self.model.eval()
        
        with open(label_map_json_path, "r", encoding="utf-8") as f:
            idx_to_label = json.load(f)
        self.idx_to_label = {int(k): v for k, v in idx_to_label.items()}

        self.extractor = MediaPipeTasksExtractor(hand_task_path, pose_task_path)
        self.frames_per_sequence = frames_per_sequence
        self.buffer = deque(maxlen=frames_per_sequence)
        self.pred_buffer = deque(maxlen=smoothing_window)
        self.min_confidence = min_confidence

    def close(self):
        self.extractor.close()

    def predict_from_frame(self, frame_bgr):
        feat = self.extractor.extract_frame_features(frame_bgr)
        self.buffer.append(feat)

        if len(self.buffer) < self.frames_per_sequence:
            return None, 0.0

        seq = np.stack(self.buffer).astype(np.float32)
        seq = np.expand_dims(seq, axis=0)
        seq = torch.FloatTensor(seq).to(DEVICE)

        with torch.no_grad():
            outputs = self.model(seq)
            probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
        
        self.pred_buffer.append(probs)

        avg_probs = np.mean(np.stack(self.pred_buffer), axis=0)
        pred_idx = int(np.argmax(avg_probs))
        conf = float(avg_probs[pred_idx])

        if conf < self.min_confidence:
            return None, conf

        return self.idx_to_label[pred_idx], conf

In [ ]:
def predict_video(video_path):
    predictor = RealTimeSignPredictor()
    cap = cv2.VideoCapture(video_path)
    predictions = []

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            label, conf = predictor.predict_from_frame(frame)
            if label is not None:
                predictions.append((label, conf))
    finally:
        cap.release()
        predictor.close()

    if not predictions:
        return None, 0.0

    labels = [x[0] for x in predictions]
    best_label = Counter(labels).most_common(1)[0][0]
    confs = [c for l, c in predictions if l == best_label]
    return best_label, float(np.mean(confs))

# Example:
# pred_label, pred_conf = predict_video("/kaggle/input/your-video/sample.mp4")
# print(pred_label, pred_conf)

In [ ]:
def run_webcam_demo():
    predictor = RealTimeSignPredictor()
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        predictor.close()
        raise RuntimeError("Could not open webcam")

    try:
        last_text = "..."
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            label, conf = predictor.predict_from_frame(frame)
            if label is not None:
                last_text = f"{label} ({conf:.2f})"

            cv2.putText(frame, last_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
            cv2.imshow("Arabic Sign Recognition", frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        predictor.close()

# Usually run this locally, not inside Kaggle:
# run_webcam_demo()